# 02 Liquidity Radar

Purpose: review the processed liquidity score, turnover, spread proxy and score components.

Data source: `data/processed/liquidity_radar.parquet` generated from the market universe and MOEX marketdata.


## Methodology Summary

The liquidity score is a first-pass diagnostic using rank-based components: value, volume, number of trades, spread proxy and volatility penalty.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from russian_markets_lab.data.io import read_processed_dataset, read_processed_metadata

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def load_dataset(name: str) -> tuple[pd.DataFrame, dict]:
    df = read_processed_dataset(name)
    metadata = read_processed_metadata(name)
    print(f'{name}: {len(df)} rows, {len(df.columns)} columns')
    if metadata:
        print('generated_at:', metadata.get('generated_at'))
        print('source:', metadata.get('source'))
    else:
        print('metadata: missing')
    return df, metadata

def show_missing(name: str) -> None:
    print(f'No processed data found for {name}. Run the corresponding CLI pipeline first.')

from russian_markets_lab.analytics.liquidity import explain_liquidity_score_components


In [ ]:
liquidity, metadata = load_dataset('liquidity_radar')
if liquidity.empty:
    show_missing('liquidity_radar')
else:
    display(liquidity.head(20))


## Top And Bottom Liquidity Scores


In [ ]:
if not liquidity.empty and 'liquidity_score' in liquidity.columns:
    columns = [c for c in ['ticker', 'avg_daily_value', 'spread_bps', 'turnover', 'realized_volatility', 'liquidity_score'] if c in liquidity.columns]
    display(liquidity.sort_values('liquidity_score', ascending=False)[columns].head(15))
    display(liquidity.sort_values('liquidity_score', ascending=True)[columns].head(15))


## Score Components And Turnover Chart


In [ ]:
if not liquidity.empty:
    components = explain_liquidity_score_components(liquidity)
    display(components.head(20))
    if {'ticker', 'turnover'}.issubset(liquidity.columns):
        fig = px.bar(liquidity.sort_values('turnover', ascending=False).head(20), x='ticker', y='turnover', title='Turnover proxy')
        fig.show()


## Key Observations

Document where liquidity is concentrated, where spread proxy is missing and where score components disagree.


## Limitations And Disclaimer

Public ISS data may omit full order book depth. The score is diagnostic and not a perfect liquidity metric.
